In [1]:
# ============================================================
# PHASE 5 - STREAMLIT APPLICATION DEVELOPMENT
# STEP 5.1 - CELL 1
# CREATE APPLICATION DIRECTORY STRUCTURE
# ============================================================

from pathlib import Path

PROJECT_DIR = Path(r"E:\SmartVision-AI")

APP_DIR = PROJECT_DIR / "streamlit_app"
PAGES_DIR = APP_DIR / "pages"
UTILS_DIR = APP_DIR / "utils"
ASSETS_DIR = APP_DIR / "assets"

# ------------------------------------------------------------
# CREATE DIRECTORIES
# ------------------------------------------------------------

directories = [
    APP_DIR,
    PAGES_DIR,
    UTILS_DIR,
    ASSETS_DIR
]

for directory in directories:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# VERIFY
# ------------------------------------------------------------

print("=" * 70)
print("SMARTVISION STREAMLIT APPLICATION STRUCTURE")
print("=" * 70)

for directory in directories:

    print(
        f"✅ {directory}"
    )

print("\nPlanned application structure:\n")

print(
r"""
SmartVision-AI/
│
├── streamlit_app/
│   ├── app.py
│   │
│   ├── pages/
│   │   ├── 1_Image_Classification.py
│   │   ├── 2_Object_Detection.py
│   │   ├── 3_Model_Performance.py
│   │   ├── 4_Webcam.py
│   │   └── 5_About.py
│   │
│   ├── utils/
│   │   ├── model_loader.py
│   │   └── inference.py
│   │
│   └── assets/
│
├── models/
├── results/
├── data/
└── notebooks/
"""
)

print("✅ Phase 5 application folders created successfully.")

SMARTVISION STREAMLIT APPLICATION STRUCTURE
✅ E:\SmartVision-AI\streamlit_app
✅ E:\SmartVision-AI\streamlit_app\pages
✅ E:\SmartVision-AI\streamlit_app\utils
✅ E:\SmartVision-AI\streamlit_app\assets

Planned application structure:


SmartVision-AI/
│
├── streamlit_app/
│   ├── app.py
│   │
│   ├── pages/
│   │   ├── 1_Image_Classification.py
│   │   ├── 2_Object_Detection.py
│   │   ├── 3_Model_Performance.py
│   │   ├── 4_Webcam.py
│   │   └── 5_About.py
│   │
│   ├── utils/
│   │   ├── model_loader.py
│   │   └── inference.py
│   │
│   └── assets/
│
├── models/
├── results/
├── data/
└── notebooks/

✅ Phase 5 application folders created successfully.


In [2]:
# ============================================================
# PHASE 5 - STEP 5.1
# CELL 2 - CREATE SHARED MODEL LOADER
# ============================================================

MODEL_LOADER_PATH = UTILS_DIR / "model_loader.py"

model_loader_code = r'''
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import models
from ultralytics import YOLO


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(__file__).resolve().parents[2]

MODELS_DIR = PROJECT_DIR / "models"

VGG16_PATH = MODELS_DIR / "vgg16_final_best.pth"
RESNET50_PATH = MODELS_DIR / "resnet50_final_best.pth"
MOBILENETV2_PATH = MODELS_DIR / "mobilenetv2_final_best.pth"
EFFICIENTNETB0_PATH = MODELS_DIR / "efficientnetb0_final_best.pth"

YOLO_PATH = (
    PROJECT_DIR
    / "runs"
    / "detection"
    / "yolov8s_v2_targeted_15ep"
    / "weights"
    / "best.pt"
)


# ============================================================
# DEVICE
# ============================================================

def get_device(prefer_gpu=True):

    if prefer_gpu and torch.cuda.is_available():
        return torch.device("cuda:0")

    return torch.device("cpu")


# ============================================================
# CHECKPOINT HELPER
# ============================================================

def _load_checkpoint(model, checkpoint_path, device):

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=False
    )

    if (
        isinstance(checkpoint, dict)
        and "model_state_dict" in checkpoint
    ):
        state_dict = checkpoint["model_state_dict"]

    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict)

    model.to(device)
    model.eval()

    return model


# ============================================================
# VGG16
# ============================================================

def load_vgg16(device):

    model = models.vgg16(weights=None)

    model.classifier = nn.Sequential(
        nn.Linear(25088, 1024),
        nn.ReLU(),
        nn.Dropout(0.5),

        nn.Linear(1024, 256),
        nn.ReLU(),
        nn.Dropout(0.3),

        nn.Linear(256, 25)
    )

    return _load_checkpoint(
        model,
        VGG16_PATH,
        device
    )


# ============================================================
# RESNET50
# ============================================================

def load_resnet50(device):

    model = models.resnet50(weights=None)

    model.fc = nn.Sequential(
        nn.Linear(2048, 512),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(512, 25)
    )

    return _load_checkpoint(
        model,
        RESNET50_PATH,
        device
    )


# ============================================================
# MOBILENETV2
# ============================================================

def load_mobilenetv2(device):

    model = models.mobilenet_v2(weights=None)

    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(1280, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, 25)
    )

    return _load_checkpoint(
        model,
        MOBILENETV2_PATH,
        device
    )


# ============================================================
# EFFICIENTNET-B0
# ============================================================

def load_efficientnetb0(device):

    model = models.efficientnet_b0(weights=None)

    model.classifier = nn.Sequential(
        nn.Dropout(0.4),

        nn.Linear(1280, 512),

        nn.BatchNorm1d(512),

        nn.SiLU(),

        nn.Dropout(0.3),

        nn.Linear(512, 25)
    )

    return _load_checkpoint(
        model,
        EFFICIENTNETB0_PATH,
        device
    )


# ============================================================
# YOLOv8s
# ============================================================

def load_yolo():

    model = YOLO(
        str(YOLO_PATH)
    )

    return model


# ============================================================
# LOAD ALL CLASSIFIERS
# ============================================================

def load_all_classifiers(prefer_gpu=True):

    device = get_device(prefer_gpu)

    models_dict = {
        "VGG16": load_vgg16(device),
        "ResNet50": load_resnet50(device),
        "MobileNetV2": load_mobilenetv2(device),
        "EfficientNetB0": load_efficientnetb0(device)
    }

    return models_dict, device
'''


MODEL_LOADER_PATH.write_text(
    model_loader_code,
    encoding="utf-8"
)


print("=" * 70)
print("SMARTVISION MODEL LOADER")
print("=" * 70)

print("\nCreated:")
print(MODEL_LOADER_PATH)

print("\nModels configured:")
print("✅ VGG16")
print("✅ ResNet50")
print("✅ MobileNetV2")
print("✅ EfficientNetB0")
print("✅ YOLOv8s final detector")

print(
    "\n✅ Shared model loader created successfully."
)

SMARTVISION MODEL LOADER

Created:
E:\SmartVision-AI\streamlit_app\utils\model_loader.py

Models configured:
✅ VGG16
✅ ResNet50
✅ MobileNetV2
✅ EfficientNetB0
✅ YOLOv8s final detector

✅ Shared model loader created successfully.


In [3]:
# ============================================================
# PHASE 5 - STEP 5.1
# CELL 3 - VERIFY MODEL LOADER AND CHECKPOINT PATHS
# ============================================================

import sys
import importlib

# ------------------------------------------------------------
# ADD STREAMLIT APP TO PYTHON PATH
# ------------------------------------------------------------

APP_DIR_STR = str(APP_DIR)

if APP_DIR_STR not in sys.path:
    sys.path.insert(0, APP_DIR_STR)


# ------------------------------------------------------------
# IMPORT MODEL LOADER
# ------------------------------------------------------------

import utils.model_loader as model_loader

importlib.reload(model_loader)


# ------------------------------------------------------------
# CHECK MODEL FILES
# ------------------------------------------------------------

model_paths = {
    "VGG16": model_loader.VGG16_PATH,
    "ResNet50": model_loader.RESNET50_PATH,
    "MobileNetV2": model_loader.MOBILENETV2_PATH,
    "EfficientNetB0": model_loader.EFFICIENTNETB0_PATH,
    "YOLOv8s": model_loader.YOLO_PATH
}


print("=" * 70)
print("SMARTVISION MODEL PATH VERIFICATION")
print("=" * 70)

all_found = True

for model_name, model_path in model_paths.items():

    exists = model_path.exists()

    status = "✅ FOUND" if exists else "❌ MISSING"

    print(
        f"{model_name:<16} : {status}"
    )

    print(
        f"{'':16}   {model_path}"
    )

    if not exists:
        all_found = False


print("\n" + "-" * 70)

if all_found:

    print("✅ All final model checkpoints found.")
    print("✅ model_loader.py imported successfully.")
    print("✅ Streamlit model configuration verified.")

else:

    print("❌ One or more model checkpoints are missing.")
    print("Do not continue until the path is corrected.")

SMARTVISION MODEL PATH VERIFICATION
VGG16            : ✅ FOUND
                   E:\SmartVision-AI\models\vgg16_final_best.pth
ResNet50         : ✅ FOUND
                   E:\SmartVision-AI\models\resnet50_final_best.pth
MobileNetV2      : ✅ FOUND
                   E:\SmartVision-AI\models\mobilenetv2_final_best.pth
EfficientNetB0   : ✅ FOUND
                   E:\SmartVision-AI\models\efficientnetb0_final_best.pth
YOLOv8s          : ✅ FOUND
                   E:\SmartVision-AI\runs\detection\yolov8s_v2_targeted_15ep\weights\best.pt

----------------------------------------------------------------------
✅ All final model checkpoints found.
✅ model_loader.py imported successfully.
✅ Streamlit model configuration verified.


In [4]:
# ============================================================
# PHASE 5 - STEP 5.1
# CELL 4 - CREATE SHARED INFERENCE MODULE
# ============================================================

INFERENCE_PATH = UTILS_DIR / "inference.py"

inference_code = r'''
import time

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from torchvision import transforms


# ============================================================
# CLASS NAMES
# ============================================================

# Classification models use ImageFolder alphabetical order
CLASSIFICATION_CLASSES = [
    "airplane",
    "bed",
    "bench",
    "bicycle",
    "bird",
    "bottle",
    "bowl",
    "bus",
    "cake",
    "car",
    "cat",
    "chair",
    "cow",
    "cup",
    "dog",
    "elephant",
    "horse",
    "motorcycle",
    "person",
    "pizza",
    "potted plant",
    "stop sign",
    "traffic light",
    "train",
    "truck"
]


# ============================================================
# CLASSIFICATION PREPROCESSING
# ============================================================

classification_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# SINGLE CLASSIFIER PREDICTION
# ============================================================

def classify_image(
    model,
    image,
    device,
    top_k=5
):
    """
    Run one classification model on a PIL image.
    Returns top-k predictions and inference latency.
    """

    image = image.convert("RGB")

    tensor = (
        classification_transform(image)
        .unsqueeze(0)
        .to(device)
    )

    start = time.perf_counter()

    with torch.inference_mode():

        logits = model(tensor)

        probabilities = torch.softmax(
            logits,
            dim=1
        )[0]

    # Synchronize before timing GPU inference
    if device.type == "cuda":
        torch.cuda.synchronize()

    latency_ms = (
        time.perf_counter() - start
    ) * 1000

    top_k = min(
        top_k,
        len(CLASSIFICATION_CLASSES)
    )

    scores, indices = torch.topk(
        probabilities,
        k=top_k
    )

    predictions = []

    for score, index in zip(
        scores.cpu().tolist(),
        indices.cpu().tolist()
    ):

        predictions.append({
            "class": CLASSIFICATION_CLASSES[index],
            "confidence": float(score)
        })

    return {
        "predicted_class": predictions[0]["class"],
        "confidence": predictions[0]["confidence"],
        "top_predictions": predictions,
        "latency_ms": latency_ms
    }


# ============================================================
# COMPARE ALL CLASSIFICATION MODELS
# ============================================================

def compare_classifiers(
    models_dict,
    image,
    device,
    top_k=5
):
    """
    Run VGG16, ResNet50, MobileNetV2 and EfficientNetB0
    on the same image.
    """

    results = {}

    for model_name, model in models_dict.items():

        results[model_name] = classify_image(
            model=model,
            image=image,
            device=device,
            top_k=top_k
        )

    return results


# ============================================================
# YOLO OBJECT DETECTION
# ============================================================

def detect_objects(
    yolo_model,
    image,
    confidence_threshold=0.50,
    nms_iou_threshold=0.70,
    device="cpu"
):
    """
    Detect multiple objects with YOLO.

    Ultralytics performs NMS internally using the supplied
    IoU threshold.
    """

    image = image.convert("RGB")

    np_image = np.array(image)

    results = yolo_model.predict(
        source=np_image,
        imgsz=640,
        conf=confidence_threshold,
        iou=nms_iou_threshold,
        device=device,
        verbose=False
    )

    result = results[0]

    detections = []

    if result.boxes is not None:

        width, height = image.size

        for box in result.boxes:

            x1, y1, x2, y2 = (
                box.xyxy[0]
                .cpu()
                .tolist()
            )

            confidence = float(
                box.conf[0].cpu()
            )

            class_id = int(
                box.cls[0].cpu()
            )

            # Refine / clip coordinates
            x1 = max(
                0,
                min(int(round(x1)), width - 1)
            )

            y1 = max(
                0,
                min(int(round(y1)), height - 1)
            )

            x2 = max(
                0,
                min(int(round(x2)), width)
            )

            y2 = max(
                0,
                min(int(round(y2)), height)
            )

            if x2 <= x1 or y2 <= y1:
                continue

            detections.append({
                "class_id": class_id,
                "class": yolo_model.names[class_id],
                "confidence": confidence,
                "bbox": [x1, y1, x2, y2]
            })

    detections.sort(
        key=lambda item: item["confidence"],
        reverse=True
    )

    return detections


# ============================================================
# OPTIONAL RESNET50 VERIFICATION
# ============================================================

def verify_detections(
    image,
    detections,
    resnet_model,
    device
):
    """
    Verify each YOLO crop using the best classification model.
    YOLO remains the primary detection label.
    """

    verified = []

    for detection in detections:

        x1, y1, x2, y2 = detection["bbox"]

        crop = image.crop(
            (x1, y1, x2, y2)
        )

        cnn_result = classify_image(
            model=resnet_model,
            image=crop,
            device=device,
            top_k=5
        )

        item = detection.copy()

        item["cnn_class"] = (
            cnn_result["predicted_class"]
        )

        item["cnn_confidence"] = (
            cnn_result["confidence"]
        )

        item["class_match"] = (
            item["class"]
            == item["cnn_class"]
        )

        verified.append(item)

    return verified


# ============================================================
# DRAW DETECTIONS
# ============================================================

def draw_detections(
    image,
    detections
):
    """
    Draw bounding boxes, labels and confidence scores.
    """

    output = image.convert("RGB").copy()

    draw = ImageDraw.Draw(output)

    for detection in detections:

        x1, y1, x2, y2 = detection["bbox"]

        label = (
            f"{detection['class']} "
            f"{detection['confidence'] * 100:.1f}%"
        )

        if "class_match" in detection:

            if detection["class_match"]:
                label += " | Verified"

            else:
                label += (
                    f" | CNN: "
                    f"{detection['cnn_class']}"
                )

        # Bounding box
        draw.rectangle(
            [x1, y1, x2, y2],
            outline="red",
            width=3
        )

        # Text background
        text_bbox = draw.textbbox(
            (x1, y1),
            label
        )

        text_height = (
            text_bbox[3] - text_bbox[1]
        )

        label_y = max(
            0,
            y1 - text_height - 8
        )

        text_bbox = draw.textbbox(
            (x1, label_y),
            label
        )

        draw.rectangle(
            [
                text_bbox[0] - 3,
                text_bbox[1] - 3,
                text_bbox[2] + 3,
                text_bbox[3] + 3
            ],
            fill="red"
        )

        draw.text(
            (x1, label_y),
            label,
            fill="white"
        )

    return output
'''


INFERENCE_PATH.write_text(
    inference_code,
    encoding="utf-8"
)


print("=" * 70)
print("SMARTVISION INFERENCE MODULE")
print("=" * 70)

print("\nCreated:")
print(INFERENCE_PATH)

print("\nFunctions configured:")
print("✅ Classification preprocessing")
print("✅ Single-model classification")
print("✅ Four-model comparison")
print("✅ Top-5 predictions")
print("✅ YOLO multi-object detection")
print("✅ NMS configuration")
print("✅ Confidence filtering")
print("✅ Optional ResNet50 verification")
print("✅ Bounding-box visualization")

print(
    "\n✅ Shared inference module created successfully."
)

SMARTVISION INFERENCE MODULE

Created:
E:\SmartVision-AI\streamlit_app\utils\inference.py

Functions configured:
✅ Classification preprocessing
✅ Single-model classification
✅ Four-model comparison
✅ Top-5 predictions
✅ YOLO multi-object detection
✅ NMS configuration
✅ Confidence filtering
✅ Optional ResNet50 verification
✅ Bounding-box visualization

✅ Shared inference module created successfully.


In [5]:
# ============================================================
# PHASE 5 - STEP 5.1
# CELL 5 - TEST SHARED YOLO INFERENCE MODULE
# ============================================================

import importlib
from PIL import Image

import utils.inference as inference
importlib.reload(inference)


print("=" * 70)
print("SMARTVISION STREAMLIT BACKEND TEST")
print("=" * 70)


# ------------------------------------------------------------
# LOAD FINAL YOLO MODEL
# ------------------------------------------------------------

test_yolo = model_loader.load_yolo()

print("\n✅ Final YOLOv8s loaded.")


# ------------------------------------------------------------
# SELECT TEST IMAGE
# ------------------------------------------------------------

TEST_IMAGE_PATH = (
    PROJECT_DIR
    / "data"
    / "optimized_detection_v2"
    / "images"
    / "test"
    / "coco_000005.jpg"
)

test_image = Image.open(
    TEST_IMAGE_PATH
).convert("RGB")

print("✅ Test image loaded:")
print(TEST_IMAGE_PATH.name)


# ------------------------------------------------------------
# RUN DETECTION THROUGH NEW MODULE
# ------------------------------------------------------------

detections = inference.detect_objects(
    yolo_model=test_yolo,
    image=test_image,
    confidence_threshold=0.50,
    nms_iou_threshold=0.70,
    device="cpu"
)

print(
    "\nDetections:",
    len(detections)
)

for i, detection in enumerate(
    detections,
    start=1
):

    print(
        f"{i}. "
        f"{detection['class']} | "
        f"{detection['confidence'] * 100:.2f}% | "
        f"{detection['bbox']}"
    )


# ------------------------------------------------------------
# TEST VISUALIZATION
# ------------------------------------------------------------

visualized_image = inference.draw_detections(
    image=test_image,
    detections=detections
)

STREAMLIT_TEST_OUTPUT = (
    PROJECT_DIR
    / "results"
    / "integration"
    / "phase5_backend_test.png"
)

visualized_image.save(
    STREAMLIT_TEST_OUTPUT
)


print("\nSaved visualization:")
print(STREAMLIT_TEST_OUTPUT)

print("\n✅ inference.py imported successfully.")
print("✅ YOLO detection function verified.")
print("✅ Bounding-box visualization verified.")
print("✅ Streamlit backend test complete.")

SMARTVISION STREAMLIT BACKEND TEST

✅ Final YOLOv8s loaded.
✅ Test image loaded:
coco_000005.jpg

Detections: 1
1. dog | 84.82% | [223, 41, 570, 310]

Saved visualization:
E:\SmartVision-AI\results\integration\phase5_backend_test.png

✅ inference.py imported successfully.
✅ YOLO detection function verified.
✅ Bounding-box visualization verified.
✅ Streamlit backend test complete.


In [6]:
# ============================================================
# PHASE 5 - STEP 5.2
# CELL 6 - CREATE STREAMLIT HOME PAGE
# ============================================================

APP_FILE = APP_DIR / "app.py"

app_code = r'''
import streamlit as st


# ============================================================
# PAGE CONFIGURATION
# ============================================================

st.set_page_config(
    page_title="SmartVision AI",
    page_icon="👁️",
    layout="wide",
    initial_sidebar_state="expanded"
)


# ============================================================
# HEADER
# ============================================================

st.title("👁️ SmartVision AI")

st.subheader(
    "Intelligent Multi-Class Object Recognition System"
)

st.markdown(
    """
    SmartVision AI is a deep-learning computer vision
    application for **image classification** and
    **multi-object detection**.

    The system combines multiple CNN classification models
    with a fine-tuned YOLOv8 object detector to recognize
    objects from **25 selected COCO classes**.
    """
)

st.divider()


# ============================================================
# PROJECT FEATURES
# ============================================================

st.header("🚀 Key Features")

col1, col2, col3 = st.columns(3)

with col1:

    st.subheader("🖼️ Image Classification")

    st.write(
        """
        Compare predictions from four transfer-learning
        CNN architectures:
        """
    )

    st.markdown(
        """
        - VGG16
        - ResNet50
        - MobileNetV2
        - EfficientNetB0
        """
    )


with col2:

    st.subheader("🎯 Object Detection")

    st.write(
        """
        Detect multiple objects in an uploaded image using
        the fine-tuned YOLOv8s detector.
        """
    )

    st.markdown(
        """
        - Bounding boxes
        - Class labels
        - Confidence scores
        - Adjustable confidence threshold
        """
    )


with col3:

    st.subheader("📊 Model Analysis")

    st.write(
        """
        Explore classification and detection performance
        through evaluation metrics and visualizations.
        """
    )

    st.markdown(
        """
        - Accuracy
        - Precision / Recall / F1
        - Top-5 accuracy
        - mAP metrics
        - Inference performance
        """
    )


st.divider()


# ============================================================
# SYSTEM WORKFLOW
# ============================================================

st.header("⚙️ How SmartVision Works")

st.markdown(
    """
    **Classification workflow**

    Upload Image → Preprocessing → Select CNN Model →
    Top-5 Predictions → Confidence Scores

    **Detection workflow**

    Upload Image → YOLOv8s Detection → NMS →
    Confidence Filtering → Bounding Boxes →
    Optional ResNet50 Verification
    """
)


st.divider()


# ============================================================
# DATASET INFORMATION
# ============================================================

st.header("📚 Dataset")

col1, col2, col3 = st.columns(3)

with col1:
    st.metric(
        "Object Classes",
        "25"
    )

with col2:
    st.metric(
        "Classification Images",
        "2,500"
    )

with col3:
    st.metric(
        "Detection Images",
        "1,817"
    )


st.caption(
    "Dataset derived from a curated subset of COCO."
)


st.divider()


# ============================================================
# USER INSTRUCTIONS
# ============================================================

st.header("🧭 Getting Started")

st.markdown(
    """
    Use the navigation menu in the sidebar to explore
    SmartVision AI.

    **Image Classification**
    — upload an image and compare CNN predictions.

    **Object Detection**
    — upload an image and detect multiple objects.

    **Model Performance**
    — explore evaluation results and model comparisons.

    **Webcam**
    — optional real-time recognition interface.

    **About**
    — learn about the dataset, architectures and
    technologies used in the project.
    """
)


st.divider()


# ============================================================
# FOOTER
# ============================================================

st.caption(
    "SmartVision AI | Deep Learning Computer Vision Project"
)
'''


APP_FILE.write_text(
    app_code,
    encoding="utf-8"
)


print("=" * 70)
print("SMARTVISION STREAMLIT HOME PAGE")
print("=" * 70)

print("\nCreated:")
print(APP_FILE)

print("\nHome page sections:")
print("✅ Project overview")
print("✅ Key features")
print("✅ Classification workflow")
print("✅ Object detection workflow")
print("✅ Dataset summary")
print("✅ User instructions")
print("✅ Navigation guidance")

print(
    "\n✅ Streamlit Home page created successfully."
)

SMARTVISION STREAMLIT HOME PAGE

Created:
E:\SmartVision-AI\streamlit_app\app.py

Home page sections:
✅ Project overview
✅ Key features
✅ Classification workflow
✅ Object detection workflow
✅ Dataset summary
✅ User instructions
✅ Navigation guidance

✅ Streamlit Home page created successfully.


In [7]:
# ============================================================
# PHASE 5 - STEP 5.2
# CELL 7 - LAUNCH STREAMLIT APPLICATION
# ============================================================

import subprocess
import sys

print("=" * 70)
print("LAUNCHING SMARTVISION AI")
print("=" * 70)

print("\nStarting Streamlit...")
print("A browser window should open automatically.")
print("\nIf it does not open, use:")
print("http://localhost:8501")

streamlit_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP_FILE),
        "--server.port=8501"
    ]
)

print("\n✅ Streamlit process started.")
print("Process ID:", streamlit_process.pid)

LAUNCHING SMARTVISION AI

Starting Streamlit...
A browser window should open automatically.

If it does not open, use:
http://localhost:8501

✅ Streamlit process started.
Process ID: 10604


In [8]:
# ============================================================
# PHASE 5 - STEP 5.3
# CELL 8 - CREATE IMAGE CLASSIFICATION PAGE
# ============================================================

CLASSIFICATION_PAGE = (
    PAGES_DIR / "1_Image_Classification.py"
)

classification_page_code = r'''
import sys
from pathlib import Path

import pandas as pd
import streamlit as st
from PIL import Image


# ============================================================
# PATH SETUP
# ============================================================

APP_DIR = Path(__file__).resolve().parents[1]

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))


from utils.model_loader import (
    get_device,
    load_vgg16,
    load_resnet50,
    load_mobilenetv2,
    load_efficientnetb0
)

from utils.inference import classify_image


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Image Classification | SmartVision AI",
    page_icon="🖼️",
    layout="wide"
)


# ============================================================
# CACHED MODEL LOADERS
# ============================================================

@st.cache_resource
def get_vgg16():
    device = get_device(prefer_gpu=True)
    return load_vgg16(device), device


@st.cache_resource
def get_resnet50():
    device = get_device(prefer_gpu=True)
    return load_resnet50(device), device


@st.cache_resource
def get_mobilenetv2():
    device = get_device(prefer_gpu=True)
    return load_mobilenetv2(device), device


@st.cache_resource
def get_efficientnetb0():
    device = get_device(prefer_gpu=True)
    return load_efficientnetb0(device), device


# ============================================================
# HEADER
# ============================================================

st.title("🖼️ Image Classification")

st.write(
    """
    Upload an image and classify it using the four
    transfer-learning CNN models developed for SmartVision AI.
    """
)

st.info(
    "The models recognize 25 selected COCO object classes."
)


# ============================================================
# SIDEBAR CONTROLS
# ============================================================

st.sidebar.header("Classification Settings")

model_option = st.sidebar.selectbox(
    "Select model",
    [
        "Compare All Models",
        "ResNet50",
        "EfficientNetB0",
        "VGG16",
        "MobileNetV2"
    ]
)

top_k = st.sidebar.slider(
    "Number of predictions",
    min_value=1,
    max_value=5,
    value=5
)


# ============================================================
# IMAGE UPLOAD
# ============================================================

uploaded_file = st.file_uploader(
    "Upload an image",
    type=["jpg", "jpeg", "png"]
)


if uploaded_file is None:

    st.markdown(
        """
        ### How to use

        1. Upload a JPG, JPEG or PNG image.
        2. Select a CNN model from the sidebar.
        3. Choose how many predictions to display.
        4. SmartVision AI will show the predicted class,
           confidence score and Top-K results.
        """
    )

    st.stop()


# ============================================================
# LOAD IMAGE
# ============================================================

image = Image.open(
    uploaded_file
).convert("RGB")


left_col, right_col = st.columns(
    [1, 1.4]
)


with left_col:

    st.subheader("Uploaded Image")

    st.image(
        image,
        use_container_width=True
    )


# ============================================================
# PREDICTION HELPER
# ============================================================

def run_model(
    model_name,
    loader_function
):

    with st.spinner(
        f"Running {model_name}..."
    ):

        model, device = loader_function()

        result = classify_image(
            model=model,
            image=image,
            device=device,
            top_k=top_k
        )

    return result


# ============================================================
# SINGLE MODEL MODE
# ============================================================

if model_option != "Compare All Models":

    loader_map = {
        "VGG16": get_vgg16,
        "ResNet50": get_resnet50,
        "MobileNetV2": get_mobilenetv2,
        "EfficientNetB0": get_efficientnetb0
    }

    result = run_model(
        model_option,
        loader_map[model_option]
    )

    with right_col:

        st.subheader(
            f"{model_option} Prediction"
        )

        st.success(
            f"Prediction: "
            f"{result['predicted_class']}"
        )

        metric1, metric2 = st.columns(2)

        metric1.metric(
            "Confidence",
            f"{result['confidence'] * 100:.2f}%"
        )

        metric2.metric(
            "Inference Time",
            f"{result['latency_ms']:.2f} ms"
        )

        st.subheader(
            f"Top-{top_k} Predictions"
        )

        prediction_df = pd.DataFrame(
            result["top_predictions"]
        )

        prediction_df["confidence"] = (
            prediction_df["confidence"]
            * 100
        )

        prediction_df.columns = [
            "Class",
            "Confidence (%)"
        ]

        st.dataframe(
            prediction_df,
            use_container_width=True,
            hide_index=True
        )

        st.bar_chart(
            prediction_df.set_index(
                "Class"
            )["Confidence (%)"]
        )


# ============================================================
# COMPARE ALL MODELS
# ============================================================

else:

    st.subheader(
        "Four-Model Comparison"
    )

    loaders = {
        "VGG16": get_vgg16,
        "ResNet50": get_resnet50,
        "MobileNetV2": get_mobilenetv2,
        "EfficientNetB0": get_efficientnetb0
    }

    comparison_results = {}

    for name, loader in loaders.items():

        comparison_results[name] = (
            run_model(
                name,
                loader
            )
        )


    # --------------------------------------------------------
    # SUMMARY TABLE
    # --------------------------------------------------------

    summary_rows = []

    for name, result in (
        comparison_results.items()
    ):

        summary_rows.append({
            "Model": name,
            "Prediction":
                result["predicted_class"],

            "Confidence (%)":
                result["confidence"] * 100,

            "Inference (ms)":
                result["latency_ms"]
        })


    summary_df = pd.DataFrame(
        summary_rows
    )

    summary_df[
        "Confidence (%)"
    ] = summary_df[
        "Confidence (%)"
    ].round(2)

    summary_df[
        "Inference (ms)"
    ] = summary_df[
        "Inference (ms)"
    ].round(2)


    with right_col:

        st.subheader(
            "Prediction Summary"
        )

        st.dataframe(
            summary_df,
            use_container_width=True,
            hide_index=True
        )


    # --------------------------------------------------------
    # INDIVIDUAL TOP-K RESULTS
    # --------------------------------------------------------

    st.divider()

    st.subheader(
        f"Top-{top_k} Predictions by Model"
    )

    model_columns = st.columns(2)

    for index, (
        name,
        result
    ) in enumerate(
        comparison_results.items()
    ):

        with model_columns[index % 2]:

            st.markdown(
                f"### {name}"
            )

            st.write(
                f"**Prediction:** "
                f"{result['predicted_class']}"
            )

            st.write(
                f"**Confidence:** "
                f"{result['confidence'] * 100:.2f}%"
            )

            prediction_df = pd.DataFrame(
                result["top_predictions"]
            )

            prediction_df[
                "confidence"
            ] *= 100

            prediction_df.columns = [
                "Class",
                "Confidence (%)"
            ]

            st.dataframe(
                prediction_df,
                use_container_width=True,
                hide_index=True
            )


st.divider()

st.caption(
    "SmartVision AI | "
    "Multi-Model Image Classification"
)
'''


CLASSIFICATION_PAGE.write_text(
    classification_page_code,
    encoding="utf-8"
)


print("=" * 70)
print("SMARTVISION IMAGE CLASSIFICATION PAGE")
print("=" * 70)

print("\nCreated:")
print(CLASSIFICATION_PAGE)

print("\nFeatures:")
print("✅ Image upload")
print("✅ VGG16 prediction")
print("✅ ResNet50 prediction")
print("✅ MobileNetV2 prediction")
print("✅ EfficientNetB0 prediction")
print("✅ Compare all four models")
print("✅ Top-1 confidence")
print("✅ Top-5 predictions")
print("✅ Inference timing")
print("✅ Prediction table")
print("✅ Confidence chart")

print(
    "\n✅ Image Classification page created successfully."
)

SMARTVISION IMAGE CLASSIFICATION PAGE

Created:
E:\SmartVision-AI\streamlit_app\pages\1_Image_Classification.py

Features:
✅ Image upload
✅ VGG16 prediction
✅ ResNet50 prediction
✅ MobileNetV2 prediction
✅ EfficientNetB0 prediction
✅ Compare all four models
✅ Top-1 confidence
✅ Top-5 predictions
✅ Inference timing
✅ Prediction table
✅ Confidence chart

✅ Image Classification page created successfully.


In [9]:
# ============================================================
# PHASE 5 - STEP 5.4
# CELL 9 - CREATE OBJECT DETECTION PAGE
# ============================================================

DETECTION_PAGE = (
    PAGES_DIR / "2_Object_Detection.py"
)

detection_page_code = r'''
import sys
import time
from pathlib import Path

import pandas as pd
import streamlit as st
from PIL import Image


# ============================================================
# PATH SETUP
# ============================================================

APP_DIR = Path(__file__).resolve().parents[1]

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))


from utils.model_loader import (
    get_device,
    load_yolo,
    load_resnet50
)

from utils.inference import (
    detect_objects,
    verify_detections,
    draw_detections
)


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Object Detection | SmartVision AI",
    page_icon="🎯",
    layout="wide"
)


# ============================================================
# CACHED MODEL LOADERS
# ============================================================

@st.cache_resource
def get_yolo_model():
    return load_yolo()


@st.cache_resource
def get_resnet_model():

    device = get_device(
        prefer_gpu=True
    )

    model = load_resnet50(
        device
    )

    return model, device


# ============================================================
# HEADER
# ============================================================

st.title("🎯 Object Detection")

st.write(
    """
    Upload an image and detect multiple objects using the
    fine-tuned SmartVision YOLOv8s detector.
    """
)

st.info(
    "YOLOv8s supports the 25 selected SmartVision COCO classes."
)


# ============================================================
# SIDEBAR SETTINGS
# ============================================================

st.sidebar.header(
    "Detection Settings"
)

confidence_threshold = (
    st.sidebar.slider(
        "Confidence threshold",
        min_value=0.10,
        max_value=0.95,
        value=0.50,
        step=0.05
    )
)

nms_iou_threshold = (
    st.sidebar.slider(
        "NMS IoU threshold",
        min_value=0.10,
        max_value=0.90,
        value=0.70,
        step=0.05
    )
)

verify_with_resnet = (
    st.sidebar.checkbox(
        "Verify detections with ResNet50",
        value=False
    )
)


# ============================================================
# IMAGE UPLOAD
# ============================================================

uploaded_file = st.file_uploader(
    "Upload an image",
    type=[
        "jpg",
        "jpeg",
        "png"
    ]
)


if uploaded_file is None:

    st.markdown(
        """
        ### How to use

        1. Upload a JPG, JPEG or PNG image.
        2. Adjust the confidence threshold if required.
        3. Adjust the NMS IoU threshold.
        4. Optionally enable ResNet50 verification.
        5. SmartVision AI will display detected objects with
           bounding boxes, labels and confidence scores.
        """
    )

    st.stop()


# ============================================================
# LOAD IMAGE
# ============================================================

image = Image.open(
    uploaded_file
).convert("RGB")


# ============================================================
# LOAD YOLO
# ============================================================

with st.spinner(
    "Loading SmartVision YOLOv8s..."
):

    yolo_model = get_yolo_model()


# ============================================================
# SELECT INFERENCE DEVICE
# ============================================================

device = get_device(
    prefer_gpu=True
)

yolo_device = (
    0
    if device.type == "cuda"
    else "cpu"
)


# ============================================================
# RUN DETECTION
# ============================================================

with st.spinner(
    "Detecting objects..."
):

    start_time = time.perf_counter()

    detections = detect_objects(
        yolo_model=yolo_model,
        image=image,
        confidence_threshold=confidence_threshold,
        nms_iou_threshold=nms_iou_threshold,
        device=yolo_device
    )

    detection_time_ms = (
        time.perf_counter()
        - start_time
    ) * 1000


# ============================================================
# OPTIONAL CNN VERIFICATION
# ============================================================

if (
    verify_with_resnet
    and detections
):

    with st.spinner(
        "Verifying detections with ResNet50..."
    ):

        resnet_model, resnet_device = (
            get_resnet_model()
        )

        detections = verify_detections(
            image=image,
            detections=detections,
            resnet_model=resnet_model,
            device=resnet_device
        )


# ============================================================
# DRAW RESULTS
# ============================================================

result_image = draw_detections(
    image=image,
    detections=detections
)


# ============================================================
# MAIN RESULTS
# ============================================================

left_col, right_col = st.columns(
    [1.5, 1]
)


with left_col:

    st.subheader(
        "Detection Result"
    )

    st.image(
        result_image,
        use_container_width=True
    )


with right_col:

    st.subheader(
        "Detection Summary"
    )

    metric1, metric2 = st.columns(2)

    metric1.metric(
        "Objects Detected",
        len(detections)
    )

    metric2.metric(
        "Processing Time",
        f"{detection_time_ms:.2f} ms"
    )

    st.write(
        "**Inference device:**",
        str(device).upper()
    )

    st.write(
        "**Confidence threshold:**",
        f"{confidence_threshold:.2f}"
    )

    st.write(
        "**NMS IoU threshold:**",
        f"{nms_iou_threshold:.2f}"
    )

    st.write(
        "**ResNet50 verification:**",
        (
            "Enabled"
            if verify_with_resnet
            else "Disabled"
        )
    )


# ============================================================
# DETECTION TABLE
# ============================================================

st.divider()

st.subheader(
    "Detected Objects"
)


if not detections:

    st.warning(
        "No objects were detected above the selected "
        "confidence threshold."
    )

else:

    rows = []

    for index, detection in enumerate(
        detections,
        start=1
    ):

        row = {
            "#": index,
            "Class":
                detection["class"],

            "YOLO Confidence (%)":
                detection["confidence"]
                * 100,

            "Bounding Box":
                str(detection["bbox"])
        }

        if "cnn_class" in detection:

            row[
                "ResNet50 Prediction"
            ] = detection[
                "cnn_class"
            ]

            row[
                "ResNet50 Confidence (%)"
            ] = (
                detection[
                    "cnn_confidence"
                ]
                * 100
            )

            row[
                "Class Match"
            ] = (
                "Yes"
                if detection[
                    "class_match"
                ]
                else "No"
            )

        rows.append(row)


    detection_df = pd.DataFrame(
        rows
    )


    percentage_columns = [
        "YOLO Confidence (%)",
        "ResNet50 Confidence (%)"
    ]

    for column in percentage_columns:

        if column in detection_df.columns:

            detection_df[column] = (
                detection_df[column]
                .round(2)
            )


    st.dataframe(
        detection_df,
        use_container_width=True,
        hide_index=True
    )


# ============================================================
# PIPELINE INFORMATION
# ============================================================

with st.expander(
    "How detection works"
):

    st.markdown(
        """
        **SmartVision detection pipeline**

        Uploaded Image  
        ↓  
        YOLOv8s Object Detection  
        ↓  
        Confidence Filtering  
        ↓  
        Non-Maximum Suppression (NMS)  
        ↓  
        Bounding Box Refinement  
        ↓  
        Optional ResNet50 Verification  
        ↓  
        Final Visualization
        """
    )


st.divider()

st.caption(
    "SmartVision AI | "
    "YOLOv8s Multi-Object Detection"
)
'''


DETECTION_PAGE.write_text(
    detection_page_code,
    encoding="utf-8"
)


print("=" * 70)
print("SMARTVISION OBJECT DETECTION PAGE")
print("=" * 70)

print("\nCreated:")
print(DETECTION_PAGE)

print("\nFeatures:")
print("✅ Image upload")
print("✅ Final YOLOv8s detector")
print("✅ Multi-object detection")
print("✅ Adjustable confidence threshold")
print("✅ Adjustable NMS IoU threshold")
print("✅ Bounding-box visualization")
print("✅ Class labels")
print("✅ Confidence scores")
print("✅ Optional ResNet50 verification")
print("✅ Detection summary")
print("✅ Detection results table")
print("✅ GPU/CPU device selection")

print(
    "\n✅ Object Detection page created successfully."
)

SMARTVISION OBJECT DETECTION PAGE

Created:
E:\SmartVision-AI\streamlit_app\pages\2_Object_Detection.py

Features:
✅ Image upload
✅ Final YOLOv8s detector
✅ Multi-object detection
✅ Adjustable confidence threshold
✅ Adjustable NMS IoU threshold
✅ Bounding-box visualization
✅ Class labels
✅ Confidence scores
✅ Optional ResNet50 verification
✅ Detection summary
✅ Detection results table
✅ GPU/CPU device selection

✅ Object Detection page created successfully.


In [10]:
# ============================================================
# PHASE 5 - STEP 5.5
# CELL 10 - INSPECT FINAL PERFORMANCE ARTIFACTS
# ============================================================

from pathlib import Path

RESULTS_DIR = PROJECT_DIR / "results"

print("=" * 70)
print("SMARTVISION PERFORMANCE ARTIFACTS")
print("=" * 70)

# File types useful for the Model Performance page
extensions = {
    ".csv",
    ".json",
    ".png",
    ".jpg",
    ".jpeg"
}

result_files = sorted(
    [
        path
        for path in RESULTS_DIR.rglob("*")
        if path.is_file()
        and path.suffix.lower() in extensions
    ]
)

print(
    f"\nTotal result artifacts found: "
    f"{len(result_files)}"
)

print("\nAvailable files:\n")

for path in result_files:

    relative_path = path.relative_to(
        PROJECT_DIR
    )

    print(relative_path)

print(
    "\n✅ Performance artifact scan complete."
)

SMARTVISION PERFORMANCE ARTIFACTS

Total result artifacts found: 32

Available files:

results\classification\efficientnetb0_final_accuracy_curve.png
results\classification\efficientnetb0_final_confusion_matrix.png
results\classification\efficientnetb0_final_history.json
results\classification\efficientnetb0_final_loss_curve.png
results\classification\final_accuracy_size_tradeoff.png
results\classification\final_accuracy_speed_tradeoff.png
results\classification\final_classification_model_selection.json
results\classification\final_model_comparison.csv
results\classification\mobilenetv2_final_accuracy_curve.png
results\classification\mobilenetv2_final_confusion_matrix.png
results\classification\mobilenetv2_final_history.json
results\classification\mobilenetv2_final_loss_curve.png
results\classification\resnet50_final_accuracy_curve.png
results\classification\resnet50_final_confusion_matrix.png
results\classification\resnet50_final_history.json
results\classification\resnet50_final_loss

In [11]:
# ============================================================
# PHASE 5 - STEP 5.5
# CELL 11 - INSPECT PERFORMANCE DATA STRUCTURE
# ============================================================

import json
import pandas as pd


# ------------------------------------------------------------
# CLASSIFICATION COMPARISON
# ------------------------------------------------------------

CLASSIFICATION_CSV = (
    PROJECT_DIR
    / "results"
    / "classification"
    / "final_model_comparison.csv"
)

classification_df = pd.read_csv(
    CLASSIFICATION_CSV
)


print("=" * 70)
print("CLASSIFICATION PERFORMANCE DATA")
print("=" * 70)

print("\nColumns:")
print(
    classification_df.columns.tolist()
)

print("\nData:")
print(
    classification_df.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# FINAL YOLO V2 METRICS
# ------------------------------------------------------------

YOLO_METRICS_JSON = (
    PROJECT_DIR
    / "results"
    / "detection"
    / "yolov8s_v2_final_test_metrics.json"
)

with open(
    YOLO_METRICS_JSON,
    "r",
    encoding="utf-8"
) as file:

    yolo_metrics = json.load(file)


print("\n" + "=" * 70)
print("FINAL YOLOv8s PERFORMANCE DATA")
print("=" * 70)

print("\nTop-level keys:")

print(
    list(yolo_metrics.keys())
)

print("\nFull metric structure:")

print(
    json.dumps(
        yolo_metrics,
        indent=2
    )
)


print(
    "\n✅ Performance data structure inspection complete."
)

CLASSIFICATION PERFORMANCE DATA

Columns:
['Model', 'Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1 Score (%)', 'Top-5 Accuracy (%)', 'Parameters (M)', 'Model Size (MB)', 'Avg Inference (ms)', 'Median Inference (ms)', 'FPS']

Data:
         Model  Accuracy (%)  Precision (%)  Recall (%)  F1 Score (%)  Top-5 Accuracy (%)  Parameters (M)  Model Size (MB)  Avg Inference (ms)  Median Inference (ms)        FPS
      ResNet50     86.133333      88.186266   86.133333     86.271030           97.066667       24.569945        93.726902           11.270729                10.3471  88.725405
EfficientNetB0     84.266667      85.653308   84.266667     84.163089           97.600000        4.677269        17.842365           15.253758                16.0811  65.557615
         VGG16     81.600000      83.123148   81.600000     81.394888           97.866667       40.674649       155.161472            4.970905                 4.9074 201.170612
   MobileNetV2     79.733333      81.483068   79.733333  

In [12]:
# ============================================================
# PHASE 5 - STEP 5.5
# CELL 12 - CREATE MODEL PERFORMANCE PAGE
# ============================================================

PERFORMANCE_PAGE = (
    PAGES_DIR / "3_Model_Performance.py"
)

performance_page_code = r'''
import json
from pathlib import Path

import pandas as pd
import streamlit as st


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(__file__).resolve().parents[2]

CLASSIFICATION_DIR = (
    PROJECT_DIR
    / "results"
    / "classification"
)

DETECTION_DIR = (
    PROJECT_DIR
    / "results"
    / "detection"
)


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Model Performance | SmartVision AI",
    page_icon="📊",
    layout="wide"
)


# ============================================================
# HEADER
# ============================================================

st.title("📊 Model Performance")

st.write(
    """
    Explore the final evaluation results for the SmartVision
    image-classification and object-detection models.
    """
)


# ============================================================
# LOAD CLASSIFICATION RESULTS
# ============================================================

comparison_path = (
    CLASSIFICATION_DIR
    / "final_model_comparison.csv"
)

classification_df = pd.read_csv(
    comparison_path
)


# ============================================================
# CLASSIFICATION SECTION
# ============================================================

st.header("🖼️ Classification Performance")

st.write(
    """
    Four transfer-learning CNN architectures were evaluated
    on the same held-out classification test set.
    """
)


# ------------------------------------------------------------
# BEST MODEL SUMMARY
# ------------------------------------------------------------

best_row = classification_df.loc[
    classification_df[
        "Accuracy (%)"
    ].idxmax()
]

metric1, metric2, metric3, metric4 = (
    st.columns(4)
)

metric1.metric(
    "Selected Classifier",
    best_row["Model"]
)

metric2.metric(
    "Test Accuracy",
    f"{best_row['Accuracy (%)']:.2f}%"
)

metric3.metric(
    "F1 Score",
    f"{best_row['F1 Score (%)']:.2f}%"
)

metric4.metric(
    "Top-5 Accuracy",
    f"{best_row['Top-5 Accuracy (%)']:.2f}%"
)


# ------------------------------------------------------------
# CLASSIFICATION TABLE
# ------------------------------------------------------------

st.subheader("Model Comparison")

display_df = classification_df.copy()

numeric_columns = display_df.select_dtypes(
    include="number"
).columns

display_df[numeric_columns] = (
    display_df[numeric_columns]
    .round(2)
)

st.dataframe(
    display_df,
    use_container_width=True,
    hide_index=True
)


# ------------------------------------------------------------
# ACCURACY CHART
# ------------------------------------------------------------

st.subheader("Test Accuracy Comparison")

accuracy_chart = (
    classification_df[
        [
            "Model",
            "Accuracy (%)"
        ]
    ]
    .set_index("Model")
)

st.bar_chart(
    accuracy_chart
)


# ------------------------------------------------------------
# TRADE-OFF VISUALIZATIONS
# ------------------------------------------------------------

st.subheader(
    "Accuracy, Speed and Model Size Trade-offs"
)

trade_col1, trade_col2 = st.columns(2)

accuracy_speed_path = (
    CLASSIFICATION_DIR
    / "final_accuracy_speed_tradeoff.png"
)

accuracy_size_path = (
    CLASSIFICATION_DIR
    / "final_accuracy_size_tradeoff.png"
)


with trade_col1:

    st.markdown(
        "#### Accuracy vs Inference Speed"
    )

    if accuracy_speed_path.exists():

        st.image(
            str(accuracy_speed_path),
            use_container_width=True
        )


with trade_col2:

    st.markdown(
        "#### Accuracy vs Model Size"
    )

    if accuracy_size_path.exists():

        st.image(
            str(accuracy_size_path),
            use_container_width=True
        )


# ============================================================
# CONFUSION MATRICES
# ============================================================

st.subheader("Confusion Matrices")

confusion_paths = {
    "VGG16":
        CLASSIFICATION_DIR
        / "vgg16_final_confusion_matrix.png",

    "ResNet50":
        CLASSIFICATION_DIR
        / "resnet50_final_confusion_matrix.png",

    "MobileNetV2":
        CLASSIFICATION_DIR
        / "mobilenetv2_final_confusion_matrix.png",

    "EfficientNetB0":
        CLASSIFICATION_DIR
        / "efficientnetb0_final_confusion_matrix.png"
}


selected_confusion_model = (
    st.selectbox(
        "Select classification model",
        list(confusion_paths.keys()),
        index=1
    )
)

selected_confusion_path = (
    confusion_paths[
        selected_confusion_model
    ]
)

if selected_confusion_path.exists():

    st.image(
        str(selected_confusion_path),
        caption=(
            f"{selected_confusion_model} "
            "Confusion Matrix"
        ),
        use_container_width=True
    )


# ============================================================
# TRAINING CURVES
# ============================================================

with st.expander(
    "View Training Curves"
):

    selected_curve_model = (
        st.selectbox(
            "Select model for training curves",
            [
                "VGG16",
                "ResNet50",
                "MobileNetV2",
                "EfficientNetB0"
            ]
        )
    )

    curve_prefix = {
        "VGG16": "vgg16",
        "ResNet50": "resnet50",
        "MobileNetV2": "mobilenetv2",
        "EfficientNetB0": "efficientnetb0"
    }[
        selected_curve_model
    ]

    accuracy_curve = (
        CLASSIFICATION_DIR
        / f"{curve_prefix}_final_accuracy_curve.png"
    )

    loss_curve = (
        CLASSIFICATION_DIR
        / f"{curve_prefix}_final_loss_curve.png"
    )

    curve_col1, curve_col2 = (
        st.columns(2)
    )

    with curve_col1:

        if accuracy_curve.exists():

            st.image(
                str(accuracy_curve),
                caption="Accuracy Curve",
                use_container_width=True
            )

    with curve_col2:

        if loss_curve.exists():

            st.image(
                str(loss_curve),
                caption="Loss Curve",
                use_container_width=True
            )


# ============================================================
# OBJECT DETECTION SECTION
# ============================================================

st.divider()

st.header("🎯 Object Detection Performance")

YOLO_METRICS_PATH = (
    DETECTION_DIR
    / "yolov8s_v2_final_test_metrics.json"
)

with open(
    YOLO_METRICS_PATH,
    "r",
    encoding="utf-8"
) as file:

    yolo_metrics = json.load(file)


det1, det2, det3, det4 = (
    st.columns(4)
)

det1.metric(
    "Precision",
    f"{yolo_metrics['precision'] * 100:.2f}%"
)

det2.metric(
    "Recall",
    f"{yolo_metrics['recall'] * 100:.2f}%"
)

det3.metric(
    "mAP@0.5",
    f"{yolo_metrics['map50'] * 100:.2f}%"
)

det4.metric(
    "mAP@0.5:0.95",
    f"{yolo_metrics['map50_95'] * 100:.2f}%"
)


st.write(
    f"""
    **Test images:** {yolo_metrics['test_images']}  
    **Test object instances:** {yolo_metrics['test_instances']}  
    **Previous V1 mAP@0.5:** {yolo_metrics['previous_v1_map50'] * 100:.2f}%  
    **V2 improvement:** {yolo_metrics['map50_change_percentage_points']:.2f}
    percentage points
    """
)


# ============================================================
# DETECTION VISUALIZATIONS
# ============================================================

st.subheader("Detection Examples")

prediction_path = (
    DETECTION_DIR
    / "yolov8s_v2_final_predictions.png"
)

failure_path = (
    DETECTION_DIR
    / "yolov8s_v2_final_failure_cases.png"
)


visual_col1, visual_col2 = st.columns(2)


with visual_col1:

    st.markdown(
        "#### Prediction Examples"
    )

    if prediction_path.exists():

        st.image(
            str(prediction_path),
            use_container_width=True
        )


with visual_col2:

    st.markdown(
        "#### Failure Case Analysis"
    )

    if failure_path.exists():

        st.image(
            str(failure_path),
            use_container_width=True
        )


# ============================================================
# MODEL SELECTION SUMMARY
# ============================================================

st.divider()

st.header("🏆 Final Model Selection")

st.markdown(
    """
    **Classification:** ResNet50 was selected as the primary
    classification model based on the final held-out test
    evaluation.

    **Object Detection:** The final targeted YOLOv8s V2
    checkpoint is used for multi-object detection.

    Quantization was also evaluated during integration.
    Dynamic INT8 preserved the tested prediction but did not
    improve CPU latency on the development machine, so the
    FP32 ResNet50 model was retained for deployment.
    """
)


st.divider()

st.caption(
    "SmartVision AI | Model Evaluation Dashboard"
)
'''


PERFORMANCE_PAGE.write_text(
    performance_page_code,
    encoding="utf-8"
)


print("=" * 70)
print("SMARTVISION MODEL PERFORMANCE PAGE")
print("=" * 70)

print("\nCreated:")
print(PERFORMANCE_PAGE)

print("\nIncluded:")
print("✅ Classification comparison table")
print("✅ Accuracy comparison")
print("✅ Precision / Recall / F1 / Top-5 metrics")
print("✅ Accuracy-speed trade-off")
print("✅ Accuracy-size trade-off")
print("✅ Four confusion matrices")
print("✅ Training accuracy/loss curves")
print("✅ Final YOLO precision and recall")
print("✅ Final YOLO mAP@0.5")
print("✅ Final YOLO mAP@0.5:0.95")
print("✅ Detection prediction examples")
print("✅ Failure-case visualization")
print("✅ Final model-selection summary")

print(
    "\n✅ Model Performance page created successfully."
)

SMARTVISION MODEL PERFORMANCE PAGE

Created:
E:\SmartVision-AI\streamlit_app\pages\3_Model_Performance.py

Included:
✅ Classification comparison table
✅ Accuracy comparison
✅ Precision / Recall / F1 / Top-5 metrics
✅ Accuracy-speed trade-off
✅ Accuracy-size trade-off
✅ Four confusion matrices
✅ Training accuracy/loss curves
✅ Final YOLO precision and recall
✅ Final YOLO mAP@0.5
✅ Final YOLO mAP@0.5:0.95
✅ Detection prediction examples
✅ Failure-case visualization
✅ Final model-selection summary

✅ Model Performance page created successfully.


In [13]:
# ============================================================
# PHASE 5 - STEP 5.6
# CREATE ABOUT PAGE
# ============================================================

ABOUT_PAGE = PAGES_DIR / "5_About.py"

about_page_code = r'''
import streamlit as st


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="About | SmartVision AI",
    page_icon="ℹ️",
    layout="wide"
)


# ============================================================
# HEADER
# ============================================================

st.title("ℹ️ About SmartVision AI")

st.subheader(
    "Intelligent Multi-Class Object Recognition System"
)

st.write(
    """
    SmartVision AI is a deep-learning computer vision project
    designed for image classification and multi-object detection.

    The system combines four transfer-learning CNN architectures
    with a fine-tuned YOLOv8s object detector to recognize
    25 selected COCO object classes.
    """
)

st.divider()


# ============================================================
# PROJECT DOCUMENTATION
# ============================================================

st.header("📘 Project Overview")

st.write(
    """
    SmartVision AI demonstrates an end-to-end computer vision
    workflow covering dataset preparation, exploratory analysis,
    transfer learning, object detection, model evaluation,
    inference-pipeline development and interactive deployment.

    The application provides two primary prediction workflows:

    **Image Classification**
    - Classifies an uploaded image using trained CNN models.
    - Supports VGG16, ResNet50, MobileNetV2 and EfficientNetB0.
    - Displays Top-5 predictions and confidence scores.
    - Allows direct comparison of all four models.

    **Object Detection**
    - Detects multiple objects using the fine-tuned YOLOv8s model.
    - Displays bounding boxes, class labels and confidence scores.
    - Supports adjustable confidence and NMS IoU thresholds.
    - Provides optional ResNet50 classification verification.
    """
)


# ============================================================
# DATASET INFORMATION
# ============================================================

st.header("📚 Dataset")

col1, col2, col3 = st.columns(3)

col1.metric(
    "Object Classes",
    "25"
)

col2.metric(
    "Classification Images",
    "2,500"
)

col3.metric(
    "Detection Images",
    "1,817"
)

st.write(
    """
    The dataset was created from a curated subset of COCO.

    **Classification dataset**
    - 25 classes
    - 100 images per class
    - 2,500 object crops
    - 224 × 224 RGB images
    - 70% training / 15% validation / 15% testing

    **Detection dataset**
    - 1,817 unique source images
    - 10,988 annotated objects
    - YOLO-format bounding-box annotations
    - Leakage-free train, validation and test splits
    """
)


# ============================================================
# SUPPORTED CLASSES
# ============================================================

st.subheader("Supported Object Classes")

classes = [
    "person", "bicycle", "car", "motorcycle", "airplane",
    "bus", "train", "truck", "traffic light", "stop sign",
    "bench", "bird", "cat", "dog", "horse",
    "cow", "elephant", "bottle", "cup", "bowl",
    "pizza", "cake", "chair", "potted plant", "bed"
]

st.write(
    ", ".join(classes)
)


# ============================================================
# MODEL ARCHITECTURES
# ============================================================

st.header("🧠 Model Architectures")

model_col1, model_col2 = st.columns(2)


with model_col1:

    st.subheader("Classification Models")

    st.markdown(
        """
        **VGG16**
        - ImageNet pretrained
        - Frozen convolutional backbone
        - Custom fully connected classifier

        **ResNet50**
        - ImageNet pretrained
        - Partial fine-tuning
        - Custom classification head
        - Selected primary classifier

        **MobileNetV2**
        - Lightweight ImageNet pretrained architecture
        - Designed for efficient inference

        **EfficientNetB0**
        - ImageNet pretrained
        - Partial fine-tuning
        - Efficient compound-scaled architecture
        """
    )


with model_col2:

    st.subheader("Object Detection Model")

    st.markdown(
        """
        **YOLOv8s**

        - Fine-tuned for the 25 SmartVision classes
        - Multi-object detection
        - Bounding-box localization
        - Confidence scoring
        - Non-Maximum Suppression
        - GPU acceleration supported
        """
    )


# ============================================================
# FINAL MODEL RESULTS
# ============================================================

st.header("📊 Final Model Results")

result_col1, result_col2 = st.columns(2)


with result_col1:

    st.subheader("ResNet50 Classification")

    st.metric(
        "Test Accuracy",
        "86.13%"
    )

    st.metric(
        "F1 Score",
        "86.27%"
    )

    st.metric(
        "Top-5 Accuracy",
        "97.07%"
    )


with result_col2:

    st.subheader("YOLOv8s Detection")

    st.metric(
        "Precision",
        "86.74%"
    )

    st.metric(
        "Recall",
        "71.66%"
    )

    st.metric(
        "mAP@0.5",
        "80.78%"
    )

    st.metric(
        "mAP@0.5:0.95",
        "64.32%"
    )


# ============================================================
# TECHNICAL STACK
# ============================================================

st.header("🛠️ Technical Stack")

stack_col1, stack_col2, stack_col3 = st.columns(3)


with stack_col1:

    st.markdown(
        """
        **Deep Learning**
        - PyTorch
        - Torchvision
        - Ultralytics YOLOv8
        - Transfer Learning
        """
    )


with stack_col2:

    st.markdown(
        """
        **Data & Vision**
        - COCO
        - Hugging Face Datasets
        - PIL
        - OpenCV
        - NumPy
        - Pandas
        """
    )


with stack_col3:

    st.markdown(
        """
        **Application**
        - Streamlit
        - Python
        - Matplotlib
        - CUDA acceleration
        """
    )


# ============================================================
# SYSTEM PIPELINE
# ============================================================

st.header("⚙️ System Pipeline")

st.markdown(
    """
    **Classification**

    Image Upload → Preprocessing → CNN Model →
    Softmax Probabilities → Top-5 Predictions

    **Object Detection**

    Image Upload → YOLOv8s → Confidence Filtering →
    NMS → Bounding Boxes → Optional ResNet50 Verification →
    Visualization
    """
)


# ============================================================
# PERFORMANCE OPTIMIZATION
# ============================================================

st.header("⚡ Performance Optimization")

st.markdown(
    """
    SmartVision AI includes:

    - GPU-aware inference with CPU fallback
    - `torch.inference_mode()` for inference
    - Batch-processing support
    - Cloud-oriented memory management
    - Optional CNN verification to reduce unnecessary computation
    - Dynamic INT8 quantization experiment

    Dynamic INT8 quantization preserved the tested ResNet50
    prediction but did not improve CPU latency on the development
    machine. Therefore, FP32 ResNet50 was retained as the primary
    classifier.
    """
)


# ============================================================
# DEVELOPER
# ============================================================

st.header("👨‍💻 Developer Information")

st.write(
    """
    **Project:** SmartVision AI  
    **Project Type:** Deep Learning / Computer Vision  
    **Application:** Intelligent Multi-Class Object Recognition System
    """
)


# ============================================================
# FOOTER
# ============================================================

st.divider()

st.caption(
    "SmartVision AI | Intelligent Multi-Class Object Recognition System"
)
'''

ABOUT_PAGE.write_text(
    about_page_code,
    encoding="utf-8"
)

print("=" * 70)
print("SMARTVISION ABOUT PAGE")
print("=" * 70)

print("\nCreated:")
print(ABOUT_PAGE)

print("\nIncluded:")
print("✅ Project documentation")
print("✅ Dataset information")
print("✅ 25 supported classes")
print("✅ Classification architectures")
print("✅ YOLOv8s architecture")
print("✅ Final model results")
print("✅ Technical stack")
print("✅ System pipeline")
print("✅ Performance optimization")
print("✅ Developer information")

print("\n✅ About page created successfully.")

SMARTVISION ABOUT PAGE

Created:
E:\SmartVision-AI\streamlit_app\pages\5_About.py

Included:
✅ Project documentation
✅ Dataset information
✅ 25 supported classes
✅ Classification architectures
✅ YOLOv8s architecture
✅ Final model results
✅ Technical stack
✅ System pipeline
✅ Performance optimization
✅ Developer information

✅ About page created successfully.


In [14]:
# ============================================================
# PHASE 5 - STEP 5.6
# CELL 13 - CHECK STREAMLIT WEBRTC DEPENDENCY
# ============================================================

import importlib.util

print("=" * 70)
print("SMARTVISION LIVE WEBCAM DEPENDENCY CHECK")
print("=" * 70)

webrtc_available = (
    importlib.util.find_spec("streamlit_webrtc")
    is not None
)

av_available = (
    importlib.util.find_spec("av")
    is not None
)

print(
    "\nstreamlit-webrtc :",
    "✅ INSTALLED" if webrtc_available else "❌ NOT INSTALLED"
)

print(
    "PyAV             :",
    "✅ INSTALLED" if av_available else "❌ NOT INSTALLED"
)

if webrtc_available and av_available:
    print(
        "\n✅ Webcam dependencies are ready."
    )
else:
    print(
        "\n⚠️ Webcam dependencies need to be installed."
    )

SMARTVISION LIVE WEBCAM DEPENDENCY CHECK

streamlit-webrtc : ❌ NOT INSTALLED
PyAV             : ❌ NOT INSTALLED

⚠️ Webcam dependencies need to be installed.


In [15]:
# ============================================================
# PHASE 5 - STEP 5.6
# CELL 14 - INSTALL LIVE WEBCAM DEPENDENCIES
# ============================================================

import sys
import subprocess

print("=" * 70)
print("INSTALLING SMARTVISION WEBCAM DEPENDENCIES")
print("=" * 70)

packages = [
    "streamlit-webrtc",
    "av"
]

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        *packages
    ]
)

print("\n" + "=" * 70)
print("✅ Webcam dependencies installed successfully.")
print("=" * 70)

print("\nInstalled:")
print("✅ streamlit-webrtc")
print("✅ PyAV")

print(
    "\n⚠️ If Jupyter cannot import the new packages immediately, "
    "we will restart the kernel before continuing."
)

INSTALLING SMARTVISION WEBCAM DEPENDENCIES

✅ Webcam dependencies installed successfully.

Installed:
✅ streamlit-webrtc
✅ PyAV

⚠️ If Jupyter cannot import the new packages immediately, we will restart the kernel before continuing.


In [16]:
# ============================================================
# PHASE 5 - STEP 5.6
# CELL 15 - VERIFY WEBCAM DEPENDENCIES
# ============================================================

print("=" * 70)
print("SMARTVISION WEBCAM IMPORT VERIFICATION")
print("=" * 70)

try:
    import streamlit_webrtc
    print(
        "\nstreamlit-webrtc : ✅ READY",
        f"(version {streamlit_webrtc.__version__})"
    )
except Exception as e:
    print("\nstreamlit-webrtc : ❌ IMPORT FAILED")
    print("Error:", e)


try:
    import av
    print(
        "PyAV             : ✅ READY",
        f"(version {av.__version__})"
    )
except Exception as e:
    print("PyAV             : ❌ IMPORT FAILED")
    print("Error:", e)


print("\n✅ Webcam dependency verification complete.")

SMARTVISION WEBCAM IMPORT VERIFICATION


2026-09-28 17:45:33.209 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-28 17:45:33.212 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-28 17:45:33.213 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-28 17:45:33.215 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-28 17:45:33.623 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-28 17:45:33.625 No runtime found, using MemoryCacheStorageManager



streamlit-webrtc : ✅ READY (version 0.78.1)
PyAV             : ✅ READY (version 17.1.0)

✅ Webcam dependency verification complete.


In [17]:
# ============================================================
# PHASE 5 - STEP 5.6
# CELL 16 - CREATE LIVE WEBCAM DETECTION PAGE
# ============================================================

WEBCAM_PAGE = (
    PAGES_DIR / "4_Webcam.py"
)

webcam_page_code = r'''
import sys
import time
import threading
from pathlib import Path

import av
import cv2
import streamlit as st
import torch

from streamlit_webrtc import (
    VideoProcessorBase,
    WebRtcMode,
    webrtc_streamer
)


# ============================================================
# PATH SETUP
# ============================================================

APP_DIR = Path(__file__).resolve().parents[1]

if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))


from utils.model_loader import (
    get_device,
    load_yolo
)


# ============================================================
# PAGE CONFIG
# ============================================================

st.set_page_config(
    page_title="Live Webcam Detection | SmartVision AI",
    page_icon="📹",
    layout="wide"
)


# ============================================================
# LOAD YOLO ONCE
# ============================================================

@st.cache_resource
def get_webcam_yolo():
    return load_yolo()


yolo_model = get_webcam_yolo()

device = get_device(
    prefer_gpu=True
)

yolo_device = (
    0
    if device.type == "cuda"
    else "cpu"
)


# ============================================================
# PAGE HEADER
# ============================================================

st.title("📹 Live Webcam Detection")

st.write(
    """
    Run real-time object detection directly from your webcam
    using the fine-tuned SmartVision YOLOv8s model.
    """
)

st.info(
    """
    Click **START** below and allow browser camera permission
    when prompted.
    """
)


# ============================================================
# SIDEBAR SETTINGS
# ============================================================

st.sidebar.header(
    "Webcam Detection Settings"
)

confidence_threshold = (
    st.sidebar.slider(
        "Confidence threshold",
        min_value=0.10,
        max_value=0.95,
        value=0.50,
        step=0.05,
        key="webcam_confidence"
    )
)

nms_iou_threshold = (
    st.sidebar.slider(
        "NMS IoU threshold",
        min_value=0.10,
        max_value=0.90,
        value=0.70,
        step=0.05,
        key="webcam_nms"
    )
)


# ============================================================
# SHARED METRICS
# ============================================================

metrics = {
    "fps": 0.0,
    "latency_ms": 0.0,
    "detections": 0
}

metrics_lock = threading.Lock()


# ============================================================
# VIDEO PROCESSOR
# ============================================================

class SmartVisionVideoProcessor(
    VideoProcessorBase
):

    def __init__(self):

        self.previous_time = None


    def recv(self, frame):

        # ----------------------------------------------------
        # FRAME -> NUMPY
        # ----------------------------------------------------

        image = frame.to_ndarray(
            format="bgr24"
        )


        # ----------------------------------------------------
        # YOLO INFERENCE
        # ----------------------------------------------------

        start_time = time.perf_counter()

        results = yolo_model.predict(
            source=image,
            imgsz=640,
            conf=confidence_threshold,
            iou=nms_iou_threshold,
            device=yolo_device,
            verbose=False
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        latency_ms = (
            time.perf_counter()
            - start_time
        ) * 1000


        # ----------------------------------------------------
        # DRAW DETECTIONS
        # ----------------------------------------------------

        result = results[0]

        detection_count = 0

        if result.boxes is not None:

            detection_count = len(
                result.boxes
            )

            for box in result.boxes:

                x1, y1, x2, y2 = (
                    box.xyxy[0]
                    .cpu()
                    .tolist()
                )

                confidence = float(
                    box.conf[0].cpu()
                )

                class_id = int(
                    box.cls[0].cpu()
                )

                class_name = (
                    yolo_model.names[
                        class_id
                    ]
                )

                x1 = int(x1)
                y1 = int(y1)
                x2 = int(x2)
                y2 = int(y2)

                label = (
                    f"{class_name} "
                    f"{confidence * 100:.1f}%"
                )

                cv2.rectangle(
                    image,
                    (x1, y1),
                    (x2, y2),
                    (0, 255, 0),
                    2
                )

                cv2.putText(
                    image,
                    label,
                    (
                        x1,
                        max(25, y1 - 10)
                    ),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    (0, 255, 0),
                    2,
                    cv2.LINE_AA
                )


        # ----------------------------------------------------
        # FPS
        # ----------------------------------------------------

        current_time = time.perf_counter()

        if self.previous_time is None:

            fps = 0.0

        else:

            frame_time = (
                current_time
                - self.previous_time
            )

            fps = (
                1.0 / frame_time
                if frame_time > 0
                else 0.0
            )

        self.previous_time = current_time


        # ----------------------------------------------------
        # STORE METRICS
        # ----------------------------------------------------

        with metrics_lock:

            metrics["fps"] = fps

            metrics[
                "latency_ms"
            ] = latency_ms

            metrics[
                "detections"
            ] = detection_count


        # ----------------------------------------------------
        # DISPLAY METRICS ON VIDEO
        # ----------------------------------------------------

        cv2.putText(
            image,
            f"FPS: {fps:.1f}",
            (15, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (255, 255, 0),
            2,
            cv2.LINE_AA
        )

        cv2.putText(
            image,
            f"Latency: {latency_ms:.1f} ms",
            (15, 60),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (255, 255, 0),
            2,
            cv2.LINE_AA
        )

        cv2.putText(
            image,
            f"Objects: {detection_count}",
            (15, 90),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (255, 255, 0),
            2,
            cv2.LINE_AA
        )


        # ----------------------------------------------------
        # RETURN VIDEO FRAME
        # ----------------------------------------------------

        return av.VideoFrame.from_ndarray(
            image,
            format="bgr24"
        )


# ============================================================
# SYSTEM INFORMATION
# ============================================================

info1, info2, info3 = st.columns(3)

info1.metric(
    "Model",
    "YOLOv8s"
)

info2.metric(
    "Device",
    str(device).upper()
)

info3.metric(
    "Input Resolution",
    "640"
)


# ============================================================
# WEBCAM STREAM
# ============================================================

st.subheader("Live Detection")

webrtc_ctx = webrtc_streamer(
    key="smartvision-webcam",
    mode=WebRtcMode.SENDRECV,

    video_processor_factory=(
        SmartVisionVideoProcessor
    ),

    media_stream_constraints={
        "video": True,
        "audio": False
    },

    async_processing=True
)


# ============================================================
# INSTRUCTIONS
# ============================================================

st.divider()

st.subheader("How to Use")

st.markdown(
    """
    1. Click **START**.
    2. Allow webcam permission in your browser.
    3. Place supported objects in front of the camera.
    4. YOLOv8s detects objects in real time.
    5. Bounding boxes show class labels and confidence scores.
    6. FPS, latency and object count are displayed directly
       on the live video.
    7. Click **STOP** when finished.
    """
)


# ============================================================
# SUPPORTED CLASSES
# ============================================================

with st.expander(
    "View Supported Classes"
):

    st.write(
        """
        person, bicycle, car, motorcycle, airplane,
        bus, train, truck, traffic light, stop sign,
        bench, bird, cat, dog, horse, cow, elephant,
        bottle, cup, bowl, pizza, cake, chair,
        potted plant, bed
        """
    )


# ============================================================
# PERFORMANCE NOTE
# ============================================================

st.caption(
    "Live FPS depends on GPU/CPU performance, camera "
    "resolution, browser/WebRTC overhead and network conditions."
)
'''


WEBCAM_PAGE.write_text(
    webcam_page_code,
    encoding="utf-8"
)


print("=" * 70)
print("SMARTVISION LIVE WEBCAM PAGE")
print("=" * 70)

print("\nCreated:")
print(WEBCAM_PAGE)

print("\nFeatures:")
print("✅ Browser webcam access")
print("✅ Real-time YOLOv8s detection")
print("✅ GPU acceleration with CPU fallback")
print("✅ Live bounding boxes")
print("✅ Class labels")
print("✅ Confidence scores")
print("✅ Adjustable confidence threshold")
print("✅ Adjustable NMS IoU threshold")
print("✅ Live FPS display")
print("✅ Live inference latency")
print("✅ Live detected-object count")
print("✅ 25 supported classes")

print(
    "\n✅ Live Webcam Detection page created successfully."
)

SMARTVISION LIVE WEBCAM PAGE

Created:
E:\SmartVision-AI\streamlit_app\pages\4_Webcam.py

Features:
✅ Browser webcam access
✅ Real-time YOLOv8s detection
✅ GPU acceleration with CPU fallback
✅ Live bounding boxes
✅ Class labels
✅ Confidence scores
✅ Adjustable confidence threshold
✅ Adjustable NMS IoU threshold
✅ Live FPS display
✅ Live inference latency
✅ Live detected-object count
✅ 25 supported classes

✅ Live Webcam Detection page created successfully.


In [18]:
# ============================================================
# PHASE 5 - STEP 5.6
# CELL 17 - VERIFY LIVE WEBCAM PAGE
# ============================================================

from pathlib import Path

WEBCAM_PAGE = Path(
    r"E:\SmartVision-AI\streamlit_app\pages\4_Webcam.py"
)

print("=" * 70)
print("SMARTVISION LIVE WEBCAM PAGE VERIFICATION")
print("=" * 70)

print("\nPage exists :", WEBCAM_PAGE.exists())

if WEBCAM_PAGE.exists():
    print("Page size   :", f"{WEBCAM_PAGE.stat().st_size / 1024:.2f} KB")

    code = WEBCAM_PAGE.read_text(encoding="utf-8")

    checks = {
        "streamlit-webrtc": "webrtc_streamer" in code,
        "YOLO model": "load_yolo" in code,
        "Live frame processor": "SmartVisionVideoProcessor" in code,
        "Confidence threshold": "confidence_threshold" in code,
        "NMS threshold": "nms_iou_threshold" in code,
        "FPS calculation": "fps" in code,
        "Latency calculation": "latency_ms" in code,
        "Bounding boxes": "cv2.rectangle" in code,
        "Class labels": "class_name" in code,
        "GPU support": "cuda" in code,
    }

    print("\nFeature verification:")

    for feature, passed in checks.items():
        print(
            f"{'✅' if passed else '❌'} {feature}"
        )

    if all(checks.values()):
        print(
            "\n✅ Webcam page code verification PASSED."
        )
    else:
        print(
            "\n⚠️ Some webcam features are missing."
        )

else:
    print("\n❌ Webcam page was not found.")

SMARTVISION LIVE WEBCAM PAGE VERIFICATION

Page exists : True
Page size   : 9.58 KB

Feature verification:
✅ streamlit-webrtc
✅ YOLO model
✅ Live frame processor
✅ Confidence threshold
✅ NMS threshold
✅ FPS calculation
✅ Latency calculation
✅ Bounding boxes
✅ Class labels
✅ GPU support

✅ Webcam page code verification PASSED.


In [19]:
# ============================================================
# PHASE 5 - FINAL REQUIREMENT CHECK
# CELL 18 - INSPECT MODEL PERFORMANCE PAGE
# ============================================================

from pathlib import Path

PERFORMANCE_PAGE = Path(
    r"E:\SmartVision-AI\streamlit_app\pages\3_Model_Performance.py"
)

print("=" * 70)
print("MODEL PERFORMANCE PAGE INSPECTION")
print("=" * 70)

if not PERFORMANCE_PAGE.exists():
    print("\n❌ File not found:")
    print(PERFORMANCE_PAGE)

else:
    code = PERFORMANCE_PAGE.read_text(
        encoding="utf-8"
    )

    lines = code.splitlines()

    print("\nFile:")
    print(PERFORMANCE_PAGE)

    print("\nTotal lines:", len(lines))

    print("\n" + "=" * 70)
    print("LAST 120 LINES")
    print("=" * 70)

    start = max(
        0,
        len(lines) - 120
    )

    for line_number, line in enumerate(
        lines[start:],
        start=start + 1
    ):
        print(
            f"{line_number:4d} | {line}"
        )

MODEL PERFORMANCE PAGE INSPECTION

File:
E:\SmartVision-AI\streamlit_app\pages\3_Model_Performance.py

Total lines: 460

LAST 120 LINES
 341 |     encoding="utf-8"
 342 | ) as file:
 343 | 
 344 |     yolo_metrics = json.load(file)
 345 | 
 346 | 
 347 | det1, det2, det3, det4 = (
 348 |     st.columns(4)
 349 | )
 350 | 
 351 | det1.metric(
 352 |     "Precision",
 353 |     f"{yolo_metrics['precision'] * 100:.2f}%"
 354 | )
 355 | 
 356 | det2.metric(
 357 |     "Recall",
 358 |     f"{yolo_metrics['recall'] * 100:.2f}%"
 359 | )
 360 | 
 361 | det3.metric(
 362 |     "mAP@0.5",
 363 |     f"{yolo_metrics['map50'] * 100:.2f}%"
 364 | )
 365 | 
 366 | det4.metric(
 367 |     "mAP@0.5:0.95",
 368 |     f"{yolo_metrics['map50_95'] * 100:.2f}%"
 369 | )
 370 | 
 371 | 
 372 | st.write(
 373 |     f"""
 374 |     **Test images:** {yolo_metrics['test_images']}  
 375 |     **Test object instances:** {yolo_metrics['test_instances']}  
 376 |     **Previous V1 mAP@0.5:** {yolo_metrics['previ

In [20]:
# ============================================================
# PHASE 5 - FINAL REQUIREMENT CHECK
# CELL 19 - ADD CLASS-WISE YOLO PERFORMANCE
# ============================================================

from pathlib import Path

PERFORMANCE_PAGE = Path(
    r"E:\SmartVision-AI\streamlit_app\pages\3_Model_Performance.py"
)

code = PERFORMANCE_PAGE.read_text(
    encoding="utf-8"
)


# ------------------------------------------------------------
# SECTION TO INSERT
# ------------------------------------------------------------

classwise_section = r'''

# ============================================================
# CLASS-WISE DETECTION PERFORMANCE
# ============================================================

st.subheader("Class-wise Detection Performance")

st.write(
    """
    The table and chart below show the final YOLOv8s V2
    **mAP@0.5 for each of the 25 supported object classes**.
    This helps identify classes where the detector performs
    strongly and classes that remain more challenging.
    """
)


classwise_map50 = {
    "person": 0.815,
    "bicycle": 0.814,
    "car": 0.741,
    "motorcycle": 0.846,
    "airplane": 0.889,
    "bus": 0.954,
    "train": 0.929,
    "truck": 0.726,
    "traffic light": 0.629,
    "stop sign": 0.913,
    "bench": 0.316,
    "bird": 0.547,
    "cat": 0.995,
    "dog": 0.962,
    "horse": 0.981,
    "cow": 0.934,
    "elephant": 0.966,
    "bottle": 0.599,
    "cup": 0.809,
    "bowl": 0.786,
    "pizza": 0.883,
    "cake": 0.875,
    "chair": 0.663,
    "potted plant": 0.698,
    "bed": 0.924
}


classwise_df = pd.DataFrame(
    {
        "Class": list(
            classwise_map50.keys()
        ),
        "mAP@0.5 (%)": [
            value * 100
            for value in classwise_map50.values()
        ]
    }
)


classwise_df = (
    classwise_df
    .sort_values(
        "mAP@0.5 (%)",
        ascending=False
    )
    .reset_index(drop=True)
)


classwise_df.insert(
    0,
    "Rank",
    range(
        1,
        len(classwise_df) + 1
    )
)


class_col1, class_col2 = st.columns(
    [1, 1.4]
)


with class_col1:

    st.markdown(
        "#### Per-Class mAP@0.5"
    )

    st.dataframe(
        classwise_df,
        use_container_width=True,
        hide_index=True,
        column_config={
            "mAP@0.5 (%)":
                st.column_config.NumberColumn(
                    "mAP@0.5 (%)",
                    format="%.2f%%"
                )
        }
    )


with class_col2:

    st.markdown(
        "#### mAP@0.5 by Class"
    )

    chart_df = (
        classwise_df[
            ["Class", "mAP@0.5 (%)"]
        ]
        .sort_values(
            "mAP@0.5 (%)",
            ascending=True
        )
        .set_index("Class")
    )

    st.bar_chart(
        chart_df,
        horizontal=True,
        height=650
    )


best_class = classwise_df.iloc[0]
weakest_class = classwise_df.iloc[-1]


summary1, summary2 = st.columns(2)


summary1.metric(
    "Highest Class mAP@0.5",
    (
        f"{best_class['Class']} — "
        f"{best_class['mAP@0.5 (%)']:.2f}%"
    )
)


summary2.metric(
    "Lowest Class mAP@0.5",
    (
        f"{weakest_class['Class']} — "
        f"{weakest_class['mAP@0.5 (%)']:.2f}%"
    )
)


st.caption(
    "Per-class values are from the final YOLOv8s V2 "
    "test evaluation."
)

'''


# ------------------------------------------------------------
# INSERT BEFORE DETECTION VISUALIZATIONS
# ------------------------------------------------------------

marker = """
# ============================================================
# DETECTION VISUALIZATIONS
# ============================================================
"""


if "Class-wise Detection Performance" in code:

    print(
        "ℹ️ Class-wise performance section already exists."
    )

elif marker not in code:

    print(
        "❌ Could not find insertion point."
    )

else:

    updated_code = code.replace(
        marker,
        classwise_section + "\n" + marker,
        1
    )

    PERFORMANCE_PAGE.write_text(
        updated_code,
        encoding="utf-8"
    )

    print("=" * 70)
    print("CLASS-WISE PERFORMANCE UPDATE")
    print("=" * 70)

    print("\nUpdated:")
    print(PERFORMANCE_PAGE)

    print("\nAdded:")
    print("✅ All 25 YOLO classes")
    print("✅ Per-class mAP@0.5 table")
    print("✅ Ranked class performance")
    print("✅ Horizontal performance chart")
    print("✅ Highest-performing class")
    print("✅ Lowest-performing class")

    print(
        "\n✅ Class-wise performance requirement added successfully."
    )

CLASS-WISE PERFORMANCE UPDATE

Updated:
E:\SmartVision-AI\streamlit_app\pages\3_Model_Performance.py

Added:
✅ All 25 YOLO classes
✅ Per-class mAP@0.5 table
✅ Ranked class performance
✅ Horizontal performance chart
✅ Highest-performing class
✅ Lowest-performing class

✅ Class-wise performance requirement added successfully.


In [21]:
# ============================================================
# PHASE 5 - FINAL REQUIREMENT CHECK
# CELL 20 - FIX CLASS-WISE PERFORMANCE CHART
# ============================================================

from pathlib import Path

PERFORMANCE_PAGE = Path(
    r"E:\SmartVision-AI\streamlit_app\pages\3_Model_Performance.py"
)

code = PERFORMANCE_PAGE.read_text(
    encoding="utf-8"
)

old_chart = '''    chart_df = (
        classwise_df[
            ["Class", "mAP@0.5 (%)"]
        ]
        .sort_values(
            "mAP@0.5 (%)",
            ascending=True
        )
        .set_index("Class")
    )

    st.bar_chart(
        chart_df,
        horizontal=True,
        height=650
    )'''

new_chart = '''    import matplotlib.pyplot as plt

    chart_df = (
        classwise_df[
            ["Class", "mAP@0.5 (%)"]
        ]
        .sort_values(
            "mAP@0.5 (%)",
            ascending=True
        )
    )

    fig, ax = plt.subplots(
        figsize=(9, 10)
    )

    ax.barh(
        chart_df["Class"],
        chart_df["mAP@0.5 (%)"]
    )

    ax.set_xlabel("mAP@0.5 (%)")
    ax.set_ylabel("Object Class")
    ax.set_xlim(0, 100)

    ax.set_title(
        "YOLOv8s V2 Class-wise Detection Performance"
    )

    ax.grid(
        axis="x",
        alpha=0.25
    )

    fig.tight_layout()

    st.pyplot(
        fig,
        use_container_width=True
    )

    plt.close(fig)'''


if old_chart not in code:

    print("❌ Existing chart block was not found.")
    print("No changes were made.")

else:

    updated_code = code.replace(
        old_chart,
        new_chart,
        1
    )

    PERFORMANCE_PAGE.write_text(
        updated_code,
        encoding="utf-8"
    )

    print("=" * 70)
    print("CLASS-WISE CHART FIX")
    print("=" * 70)

    print("\nUpdated:")
    print(PERFORMANCE_PAGE)

    print("\n✅ Replaced blank Streamlit horizontal chart")
    print("✅ Added Matplotlib horizontal bar chart")
    print("✅ Displays all 25 classes")
    print("✅ mAP@0.5 scale fixed from 0–100%")

    print(
        "\n✅ Class-wise performance chart fix completed."
    )

CLASS-WISE CHART FIX

Updated:
E:\SmartVision-AI\streamlit_app\pages\3_Model_Performance.py

✅ Replaced blank Streamlit horizontal chart
✅ Added Matplotlib horizontal bar chart
✅ Displays all 25 classes
✅ mAP@0.5 scale fixed from 0–100%

✅ Class-wise performance chart fix completed.


In [22]:
# ============================================================
# PHASE 5 - FINAL REQUIREMENT CHECK
# CELL 21 - INSPECT HOME PAGE
# ============================================================

from pathlib import Path

HOME_PAGE = Path(
    r"E:\SmartVision-AI\streamlit_app\app.py"
)

print("=" * 70)
print("SMARTVISION HOME PAGE INSPECTION")
print("=" * 70)

if not HOME_PAGE.exists():

    print("\n❌ Home page not found:")
    print(HOME_PAGE)

else:

    code = HOME_PAGE.read_text(
        encoding="utf-8"
    )

    lines = code.splitlines()

    print("\nFile:")
    print(HOME_PAGE)

    print("\nTotal lines:", len(lines))

    print("\n" + "=" * 70)
    print("LAST 120 LINES")
    print("=" * 70)

    start = max(
        0,
        len(lines) - 120
    )

    for line_number, line in enumerate(
        lines[start:],
        start=start + 1
    ):

        print(
            f"{line_number:4d} | {line}"
        )

SMARTVISION HOME PAGE INSPECTION

File:
E:\SmartVision-AI\streamlit_app\app.py

Total lines: 216

LAST 120 LINES
  97 |         """
  98 |         Explore classification and detection performance
  99 |         through evaluation metrics and visualizations.
 100 |         """
 101 |     )
 102 | 
 103 |     st.markdown(
 104 |         """
 105 |         - Accuracy
 106 |         - Precision / Recall / F1
 107 |         - Top-5 accuracy
 108 |         - mAP metrics
 109 |         - Inference performance
 110 |         """
 111 |     )
 112 | 
 113 | 
 114 | st.divider()
 115 | 
 116 | 
 117 | # ============================================================
 118 | # SYSTEM WORKFLOW
 119 | # ============================================================
 120 | 
 121 | st.header("⚙️ How SmartVision Works")
 122 | 
 123 | st.markdown(
 124 |     """
 125 |     **Classification workflow**
 126 | 
 127 |     Upload Image → Preprocessing → Select CNN Model →
 128 |     Top-5 Predictions → Confiden

In [23]:
# ============================================================
# PHASE 5 - FINAL REQUIREMENT CHECK
# CELL 22 - ADD SAMPLE DEMO IMAGES TO HOME PAGE
# ============================================================

from pathlib import Path

HOME_PAGE = Path(
    r"E:\SmartVision-AI\streamlit_app\app.py"
)

code = HOME_PAGE.read_text(
    encoding="utf-8"
)


# ------------------------------------------------------------
# SAMPLE DEMO SECTION
# ------------------------------------------------------------

demo_section = r'''

# ============================================================
# SAMPLE DEMO IMAGES
# ============================================================

st.header("🖼️ Sample Demo")

st.write(
    """
    Example outputs generated by the trained SmartVision AI
    object-detection pipeline.
    """
)


PROJECT_DIR = Path(
    r"E:\SmartVision-AI"
)

demo_detection = (
    PROJECT_DIR
    / "results"
    / "integration"
    / "phase4_step4_2_multi_object_test.png"
)

demo_predictions = (
    PROJECT_DIR
    / "results"
    / "detection"
    / "yolov8s_v2_final_predictions.png"
)


demo_col1, demo_col2 = st.columns(2)


with demo_col1:

    st.markdown(
        "#### Multi-Object Detection"
    )

    if demo_detection.exists():

        st.image(
            str(demo_detection),
            caption=(
                "YOLOv8s multi-object detection with "
                "bounding boxes and confidence scores"
            ),
            use_container_width=True
        )

    else:

        st.info(
            "Multi-object demo image is not available."
        )


with demo_col2:

    st.markdown(
        "#### Final YOLO Prediction Examples"
    )

    if demo_predictions.exists():

        st.image(
            str(demo_predictions),
            caption=(
                "Prediction examples from the final "
                "YOLOv8s V2 detector"
            ),
            use_container_width=True
        )

    else:

        st.info(
            "YOLO prediction demo image is not available."
        )


st.caption(
    "These examples were generated using the trained "
    "SmartVision AI models."
)

st.divider()

'''


# ------------------------------------------------------------
# INSERT BEFORE USER INSTRUCTIONS
# ------------------------------------------------------------

marker = """
# ============================================================
# USER INSTRUCTIONS
# ============================================================
"""


if "SAMPLE DEMO IMAGES" in code:

    print(
        "ℹ️ Sample Demo section already exists."
    )

elif marker not in code:

    print(
        "❌ Could not find the insertion point."
    )

else:

    updated_code = code.replace(
        marker,
        demo_section + "\n" + marker,
        1
    )

    HOME_PAGE.write_text(
        updated_code,
        encoding="utf-8"
    )

    print("=" * 70)
    print("SMARTVISION HOME PAGE UPDATE")
    print("=" * 70)

    print("\nUpdated:")
    print(HOME_PAGE)

    print("\nAdded:")
    print("✅ Sample Demo section")
    print("✅ Multi-object detection example")
    print("✅ Final YOLO prediction examples")
    print("✅ Demo descriptions and captions")

    print(
        "\n✅ Home sample-demo requirement added successfully."
    )

SMARTVISION HOME PAGE UPDATE

Updated:
E:\SmartVision-AI\streamlit_app\app.py

Added:
✅ Sample Demo section
✅ Multi-object detection example
✅ Final YOLO prediction examples
✅ Demo descriptions and captions

✅ Home sample-demo requirement added successfully.


In [24]:
# ============================================================
# PHASE 5 - FINAL REQUIREMENT CHECK
# CELL 23 - FIX HOME PAGE PATH IMPORT
# ============================================================

from pathlib import Path

HOME_PAGE = Path(
    r"E:\SmartVision-AI\streamlit_app\app.py"
)

code = HOME_PAGE.read_text(
    encoding="utf-8"
)

if "from pathlib import Path" in code:

    print("ℹ️ pathlib.Path is already imported.")

else:

    # Add Path import before the Streamlit import
    if "import streamlit as st" in code:

        code = code.replace(
            "import streamlit as st",
            "from pathlib import Path\n\nimport streamlit as st",
            1
        )

        HOME_PAGE.write_text(
            code,
            encoding="utf-8"
        )

        print("=" * 70)
        print("HOME PAGE PATH IMPORT FIX")
        print("=" * 70)

        print("\nUpdated:")
        print(HOME_PAGE)

        print("\n✅ Added: from pathlib import Path")
        print("✅ Sample Demo paths can now be created")
        print("✅ No other Home page code changed")

        print(
            "\n✅ Path import fix completed successfully."
        )

    else:

        print(
            "❌ Could not locate 'import streamlit as st'."
        )

HOME PAGE PATH IMPORT FIX

Updated:
E:\SmartVision-AI\streamlit_app\app.py

✅ Added: from pathlib import Path
✅ Sample Demo paths can now be created
✅ No other Home page code changed

✅ Path import fix completed successfully.
